# 🕳️ Pothole Segmentation with UNet++ & EfficientNet-B5

> **Semantic segmentation pipeline** for detecting potholes in road images.

| Component | Detail |
|---|---|
| **Architecture** | UNet++ (Segmentation Models PyTorch) |
| **Encoder** | EfficientNet-B5 (ImageNet pre-trained) |
| **Loss** | BCE + Dice + Focal (weighted combination) |
| **Augmentation** | Albumentations (shift-scale-rotate, color jitter, noise, dropout) |
| **Training** | Mixed-precision (AMP), CosineAnnealing LR, stratified split |
| **Inference** | Multi-scale TTA + morphological post-processing |

---

## 0 · Install Dependencies

> **Note:** Run this cell once to install / upgrade required packages.

In [ ]:
!pip -q install -U albumentations segmentation-models-pytorch timm --quiet

## 1 · Imports

In [ ]:
import os
import glob
import random

import numpy as np
import cv2
from tqdm import tqdm
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

print(f"PyTorch  : {torch.__version__}")
print(f"Device   : {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 2 · Configuration & Hyperparameters

All tunable settings are gathered in one place for easy experimentation.

In [ ]:
# ── Dataset Paths (relative — run notebook from project root) ─────
TRAIN_IMG_DIR = "dataset/train/images"
TRAIN_MSK_DIR = "dataset/train/mask"
TEST_IMG_DIR  = "dataset/test/images"

# ── Reproducibility ───────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Device ─────────────────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"
n_gpus = torch.cuda.device_count()
print(f"Device: {device} | GPUs: {n_gpus}")

# ── Training Hyperparameters ──────────────────────────────────────
IMG_SIZE = 768          # Input resolution (reduce to 512 if OOM)
BATCH    = 8            # Batch size       (reduce to 4–6 if OOM)
EPOCHS   = 25
LR       = 4e-4
WD       = 1e-3
LR_MIN   = 4e-6

## 3 · Utility Functions

### 3.1 · RLE Encoder

Converts a binary mask into Run-Length Encoding for submission.

In [ ]:
def encode_rle(mask: np.ndarray, pos_value: int = 255) -> str:
    """Encode a binary mask as an RLE string."""
    binary = (mask == pos_value).astype(np.uint8)
    if binary.sum() == 0:
        return ""
    pixels = binary.T.flatten()
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[0::2]
    return " ".join(str(x) for x in runs)

### 3.2 · Post-Processing

Morphological closing + small-component removal to clean up raw predictions.

In [ ]:
def post_process_mask(bin_mask_255, min_area=200, k=5):
    """Clean binary mask with morphological closing and component filtering."""
    kernel = np.ones((k, k), np.uint8)
    closed = cv2.morphologyEx(bin_mask_255, cv2.MORPH_CLOSE, kernel)

    num, labels, stats, _ = cv2.connectedComponentsWithStats(
        (closed > 0).astype(np.uint8), connectivity=8
    )
    out = np.zeros_like(bin_mask_255)
    for i in range(1, num):
        if stats[i, cv2.CC_STAT_AREA] >= min_area:
            out[labels == i] = 255
    return out

## 4 · Data Preparation

### 4.1 · Build Image–Mask Pairs

In [ ]:
train_imgs = sorted(glob.glob(os.path.join(TRAIN_IMG_DIR, "*")))

def mask_path_from_image(img_path):
    """Derive mask path from image path.  train_326.jpg → mask_326.png"""
    base = os.path.basename(img_path)
    idx  = os.path.splitext(base)[0].split("_")[-1]
    return os.path.join(TRAIN_MSK_DIR, f"mask_{idx}.png")

pairs = [(p, mask_path_from_image(p)) for p in train_imgs]
pairs = [(i, m) for i, m in pairs if os.path.exists(m)]
print(f"Train pairs: {len(pairs)} | example: {pairs[0]}")

### 4.2 · Stratified Split by Mask Area

Bins each mask by pothole area ratio, then samples ~10 % for validation
from each bin to keep class balance.

In [ ]:
def mask_area_bin(mask_path):
    """Assign mask to area bin: 0=empty, 1=tiny, 2=small, 3=large."""
    m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    area = (m > 0).mean()
    if area < 0.001: return 0
    if area < 0.01:  return 1
    if area < 0.05:  return 2
    return 3

bins = [mask_area_bin(m) for _, m in pairs]
idxs = list(range(len(pairs)))
random.shuffle(idxs)

bin_to_idxs = {}
for i in idxs:
    bin_to_idxs.setdefault(bins[i], []).append(i)

val_idxs = []
for b, li in bin_to_idxs.items():
    k = max(1, int(0.1 * len(li)))
    val_idxs += li[:k]

val_set     = set(val_idxs)
train_idxs  = [i for i in idxs if i not in val_set]

train_pairs = [pairs[i] for i in train_idxs]
val_pairs   = [pairs[i] for i in val_idxs]
print(f"Split → train: {len(train_pairs)}  val: {len(val_pairs)}")

## 5 · Augmentations

Letterbox resize + strong augmentations for training; clean resize for validation.

In [ ]:
train_tfms = A.Compose([
    A.LongestMaxSize(max_size=IMG_SIZE),
    A.PadIfNeeded(min_height=IMG_SIZE, min_width=IMG_SIZE,
                  border_mode=cv2.BORDER_CONSTANT),

    A.Affine(shift_limit=0.06, scale_limit=0.20, rotate_limit=15,
             p=0.7, border_mode=cv2.BORDER_CONSTANT),
    A.RandomBrightnessContrast(p=0.6),
    A.HueSaturationValue(p=0.3),

    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    A.MotionBlur(blur_limit=5, p=0.15),
    A.GaussNoise(std_range=(0.04, 0.14), p=0.2),

    A.CoarseDropout(
        num_holes_range=(1, 8),
        hole_height_range=(1, IMG_SIZE // 12),
        hole_width_range=(1, IMG_SIZE // 12),
        p=0.2,
    ),

    A.Normalize(),
    ToTensorV2(),
])

val_tfms = A.Compose([
    A.LongestMaxSize(max_size=IMG_SIZE),
    A.PadIfNeeded(min_height=IMG_SIZE, min_width=IMG_SIZE,
                  border_mode=cv2.BORDER_CONSTANT),
    A.Normalize(),
    ToTensorV2(),
])

## 6 · Dataset

In [ ]:
class PotholeDataset(Dataset):
    def __init__(self, pairs, tfms):
        self.pairs = pairs
        self.tfms  = tfms

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, msk_path = self.pairs[idx]
        img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        msk = (cv2.imread(msk_path, cv2.IMREAD_GRAYSCALE) > 0).astype(np.uint8)  # 0/1

        aug = self.tfms(image=img, mask=msk)
        x = aug["image"]                        # float32 (C, H, W)
        y = aug["mask"].unsqueeze(0).float()     # (1, H, W)
        return x, y, os.path.basename(img_path)

## 7 · DataLoaders

In [ ]:
train_ds = PotholeDataset(train_pairs, train_tfms)
val_ds   = PotholeDataset(val_pairs,   val_tfms)

train_loader = DataLoader(
    train_ds, batch_size=BATCH, shuffle=True,
    num_workers=2, pin_memory=True, drop_last=True,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH, shuffle=False,
    num_workers=2, pin_memory=True,
)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

## 8 · Model Architecture

**UNet++** with **EfficientNet-B5** encoder (ImageNet pre-trained).

In [ ]:
model = smp.UnetPlusPlus(
    encoder_name="efficientnet-b5",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    activation=None,
).to(device)

if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")

## 9 · Loss Functions & Metrics

Composite loss: **0.3 × BCE + 0.5 × Dice + 0.2 × Focal**

In [ ]:
bce = nn.BCEWithLogitsLoss()

def dice_loss_from_logits(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    num = 2 * (probs * targets).sum(dim=(2, 3)) + eps
    den = (probs + targets).sum(dim=(2, 3)) + eps
    return 1 - (num / den).mean()

def focal_loss_with_logits(logits, targets, alpha=0.8, gamma=2.0):
    prob = torch.sigmoid(logits)
    pt   = torch.where(targets == 1, prob, 1 - prob)
    w    = torch.where(targets == 1, alpha, 1 - alpha)
    return (-w * (1 - pt) ** gamma * torch.log(pt + 1e-7)).mean()

def total_loss(logits, targets):
    return (0.3 * bce(logits, targets)
          + 0.5 * dice_loss_from_logits(logits, targets)
          + 0.2 * focal_loss_with_logits(logits, targets))

@torch.no_grad()
def dice_score_from_logits(logits, targets, thr=0.5, eps=1e-6):
    preds = (torch.sigmoid(logits) > thr).float()
    inter = (preds * targets).sum(dim=(2, 3))
    return ((2 * inter + eps) / (preds.sum(dim=(2, 3)) + targets.sum(dim=(2, 3)) + eps)).mean().item()

## 10 · Optimizer, Scheduler & Mixed-Precision

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR_MIN)
scaler    = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))

## 11 · Training Utilities

### 11.1 · Save / Load (DataParallel-safe)

In [ ]:
best_path = "best_model.pt"
best_dice = -1

def save_best(model, path):
    sd = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
    torch.save(sd, path)

def load_best(model, path, device):
    sd = torch.load(path, map_location=device, weights_only=True)
    if isinstance(model, nn.DataParallel):
        model.module.load_state_dict(sd)
    else:
        model.load_state_dict(sd)

### 11.2 · Visual Overlay Monitor

In [ ]:
@torch.no_grad()
def show_val_overlays(model, val_ds, n=4, thr=0.5):
    """Display image / GT / probability / prediction / overlay for n samples."""
    model.eval()
    idxs = np.linspace(0, len(val_ds) - 1, n, dtype=int)

    fig, axes = plt.subplots(n, 5, figsize=(14, 3.5 * n))
    if n == 1:
        axes = axes[np.newaxis, :]

    for row, idx in enumerate(idxs):
        x, y, name = val_ds[idx]
        logits = model(x.unsqueeze(0).to(device))
        if isinstance(logits, (list, tuple)):
            logits = logits[0]
        prob = torch.sigmoid(logits)[0, 0].cpu().numpy()

        img_disp = x.permute(1, 2, 0).cpu().numpy()
        img_disp = (img_disp - img_disp.min()) / (img_disp.max() - img_disp.min() + 1e-6)
        gt = y[0].cpu().numpy()
        pr = (prob > thr).astype(np.float32)

        overlay = img_disp.copy()
        overlay[..., 0] = np.clip(overlay[..., 0] + 0.6 * pr, 0, 1)   # red  = pred
        overlay[..., 1] = np.clip(overlay[..., 1] + 0.6 * gt, 0, 1)   # green = GT

        titles = [name, "GT", "Prob", "Pred", "Overlay (G=GT, R=Pred)"]
        imgs   = [img_disp, gt, prob, pr, overlay]
        cmaps  = [None, "gray", "jet", "gray", None]

        for col in range(5):
            axes[row, col].imshow(imgs[col], cmap=cmaps[col])
            axes[row, col].set_title(titles[col], fontsize=9)
            axes[row, col].axis("off")

    plt.tight_layout()
    plt.show()

## 12 · Training Loop

Full training with mixed-precision, cosine LR scheduling, best-model
checkpointing, and periodic visual monitoring.

In [ ]:
VAL_THR = 0.5

for epoch in range(1, EPOCHS + 1):
    # ── Train ──────────────────────────────────────────────────────
    model.train()
    tr_loss = 0.0

    for x, y, _ in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [train]"):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=(device == "cuda")):
            logits = model(x)
            loss   = total_loss(logits, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        tr_loss += loss.item()

    scheduler.step()
    tr_loss /= len(train_loader)

    # ── Validate ───────────────────────────────────────────────────
    model.eval()
    va_loss, va_dice = 0.0, 0.0

    with torch.no_grad():
        for x, y, _ in tqdm(val_loader, desc=f"Epoch {epoch}/{EPOCHS} [val]"):
            x, y = x.to(device), y.to(device)
            logits = model(x)
            va_loss += total_loss(logits, y).item()
            va_dice += dice_score_from_logits(logits, y, thr=VAL_THR)

    va_loss /= len(val_loader)
    va_dice /= len(val_loader)

    print(f"[Epoch {epoch}]  train_loss={tr_loss:.4f}  "
          f"val_loss={va_loss:.4f}  val_dice@{VAL_THR:.2f}={va_dice:.4f}")

    if va_dice > best_dice:
        best_dice = va_dice
        save_best(model, best_path)
        print(f"  ✅ New best saved: {best_path} | dice={best_dice:.4f}")

    # Show a few overlays every 5 epochs
    if epoch % 5 == 0 or epoch == 1:
        show_val_overlays(model, val_ds, n=4, thr=VAL_THR)

## 13 · Threshold Tuning

Search for the binarization threshold that maximizes Dice on the validation set.

In [ ]:
@torch.no_grad()
def eval_thresholds(model, val_loader, thresholds=np.linspace(0.25, 0.80, 12)):
    model.eval()
    scores = []
    for thr in thresholds:
        d = 0.0
        for x, y, _ in val_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            d += dice_score_from_logits(logits, y, thr=float(thr))
        scores.append(d / len(val_loader))
    best_i = int(np.argmax(scores))
    return float(thresholds[best_i]), float(scores[best_i]), list(zip(thresholds, scores))

load_best(model, best_path, device)
best_thr, best_thr_dice, all_scores = eval_thresholds(model, val_loader)
print(f"✅ Best threshold: {best_thr:.3f} | Dice: {best_thr_dice:.4f}")

## 14 · Inference with Test-Time Augmentation (TTA)

Multi-scale + horizontal flip TTA, followed by post-processing and RLE encoding.

In [ ]:
@torch.no_grad()
def predict_prob_tta(model, img_bgr, scales=(0.85, 1.0, 1.15)):
    """Predict probability map with multi-scale + hflip TTA."""
    h0, w0 = img_bgr.shape[:2]
    img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    probs_all = []

    for sc in scales:
        img_sc = cv2.resize(img, (int(w0 * sc), int(h0 * sc)),
                            interpolation=cv2.INTER_LINEAR) if sc != 1.0 else img
        hs, ws = img_sc.shape[:2]
        aug = val_tfms(image=img_sc, mask=np.zeros((hs, ws), dtype=np.uint8))
        x = aug["image"].unsqueeze(0).to(device)

        # Original
        logits1 = model(x)
        if isinstance(logits1, (list, tuple)):
            logits1 = logits1[0]
        p1 = torch.sigmoid(logits1)[0, 0].cpu().numpy()

        # Horizontal flip
        logits2 = model(torch.flip(x, dims=[3]))
        if isinstance(logits2, (list, tuple)):
            logits2 = logits2[0]
        p2 = np.flip(torch.sigmoid(logits2)[0, 0].cpu().numpy(), axis=1)

        p = 0.5 * (p1 + p2)
        probs_all.append(cv2.resize(p, (w0, h0), interpolation=cv2.INTER_LINEAR))

    return np.mean(probs_all, axis=0)

### 14.1 · Generate Submission

In [ ]:
test_ids = sorted([f for f in os.listdir(TEST_IMG_DIR) if f.lower().endswith(".jpg")])
MIN_AREA = 200  # minimum connected component area (pixels)
rows = []

for img_id in tqdm(test_ids, desc="Inference"):
    img_bgr  = cv2.imread(os.path.join(TEST_IMG_DIR, img_id))
    prob     = predict_prob_tta(model, img_bgr)
    bin_mask = (prob > best_thr).astype(np.uint8) * 255
    bin_mask = post_process_mask(bin_mask, min_area=MIN_AREA, k=5)
    rows.append({"ImageId": img_id, "rle": encode_rle(bin_mask, pos_value=255)})

sub = pd.DataFrame(rows, columns=["ImageId", "rle"])
sub.to_csv("submission.csv", index=False)
print(f"✅ Saved submission.csv  ({len(sub)} rows)")
sub.head()